## 1. ติดตั้ง Dependencies

In [ ]:

!pip install faiss-cpu numpy

   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
    --------------------------------------- 0.3/18.9 MB ? eta -:--:--
   -- ------------------------------------- 1.0/18.9 MB 3.1 MB/s eta 0:00:06
   --- ------------------------------------ 1.8/18.9 MB 3.2 MB/s eta 0:00:06
   ----- ---------------------------------- 2.6/18.9 MB 3.4 MB/s eta 0:00:05
   ------- -------------------------------- 3.4/18.9 MB 3.5 MB/s eta 0:00:05
   -------- ------------------------------- 4.2/18.9 MB 3.6 MB/s eta 0:00:05
   ---------- ----------------------------- 5.0/18.9 MB 3.6 MB/s eta 0:00:04
   ------------ --------------------------- 5.8/18.9 MB 3.7 MB/s eta 0:00:04
   ------------- -------------------------- 6.6/18.9 MB 3.7 MB/s eta 0:00:04
   --------------- ------------------------ 7.3/18.9 MB 3.7 MB/s eta 0:00:04
   ----------------- ---------------------- 8.1/18.9 MB 3.6 MB/s eta 0:00:03
   ------------------ --------------------- 8.7/18.9 MB 3.6 MB/s eta 0:00:03
   ----------

## 2. Import Libraries และโหลดข้อมูล

In [2]:
import faiss
import numpy as np
import json
from pathlib import Path

print(f"FAISS version: {faiss.__version__}")
print(f"NumPy version: {np.__version__}")

FAISS version: 1.13.2
NumPy version: 2.3.5


In [3]:
embeddings_path = Path("tourism_embeddings.npy")
metadata_path   = Path("tourism_metadata.json")

assert embeddings_path.exists(), "ไม่พบ tourism_embeddings.npy — รัน qwen3_embedding_tourism.ipynb ก่อน"
assert metadata_path.exists(),   "ไม่พบ tourism_metadata.json — รัน qwen3_embedding_tourism.ipynb ก่อน"

embeddings = np.load(str(embeddings_path)).astype(np.float32)
with open(metadata_path, encoding="utf-8") as f:
    metadata = json.load(f)

print(f"Embeddings shape : {embeddings.shape}")
print(f"จำนวน documents  : {len(metadata)}")
print(f"Embedding dim    : {embeddings.shape[1]}")

# ตรวจสอบว่า L2-normalized แล้วหรือยัง
norms = np.linalg.norm(embeddings, axis=1)
print(f"\nL2 norm (min/max): {norms.min():.4f} / {norms.max():.4f}")
print("✓ Embeddings เป็น unit vectors แล้ว" if np.allclose(norms, 1.0, atol=1e-3) else "⚠ Embeddings ยังไม่ normalized")

Embeddings shape : (54, 1024)
จำนวน documents  : 54
Embedding dim    : 1024

L2 norm (min/max): 1.0000 / 1.0000
✓ Embeddings เป็น unit vectors แล้ว


## 3. Build FAISS Index

In [4]:
EMBEDDING_DIM = embeddings.shape[1]  # 1024 สำหรับ BAAI/bge-m3

# L2-normalize อีกครั้งเพื่อความปลอดภัย (idempotent)
faiss.normalize_L2(embeddings)

# สร้าง index แบบ Flat (exact search, ไม่มี approximation)
index = faiss.IndexFlatIP(EMBEDDING_DIM)

# เพิ่ม vectors ทั้งหมดเข้า index
index.add(embeddings)

print(f"Index type  : {type(index).__name__}")
print(f"Total vectors: {index.ntotal}")
print(f"Dimension   : {index.d}")
print(f"Is trained  : {index.is_trained}")

Index type  : IndexFlatIP
Total vectors: 54
Dimension   : 1024
Is trained  : True


## 4. ตรวจสอบ Index (Sanity Check)

In [5]:
# ทดสอบ: ใช้ vector ของ doc แรกเป็น query — ควรได้ตัวเองเป็น #1 (score=1.0)
test_vec = embeddings[0:1].copy()
scores, indices = index.search(test_vec, k=3)

print("Sanity check — ค้นหาด้วย vector ของเอกสารแรก:")
for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
    name = metadata[idx]["name"]
    print(f"  #{rank} [score={score:.4f}] index={idx} → {name}")

assert indices[0][0] == 0 and abs(scores[0][0] - 1.0) < 1e-4, "❌ Sanity check ล้มเหลว"
print("\n✓ Index ทำงานถูกต้อง")

Sanity check — ค้นหาด้วย vector ของเอกสารแรก:
  #1 [score=1.0000] index=0 → วัดปทุมวนารามราชวรวิหาร
  #2 [score=0.7263] index=2 → วัดธาตุทอง
  #3 [score=0.7070] index=6 → SEA LIFE Bangkok Ocean World

✓ Index ทำงานถูกต้อง


## 5. บันทึก Index (Export)

In [ ]:
INDEX_PATH    = "tourism_faiss.index"
METADATA_PATH = "tourism_metadata.json"

faiss.write_index(index, INDEX_PATH)

index_size_kb = Path(INDEX_PATH).stat().st_size / 1024
print(f"บันทึกสำเร็จ:")
print(f"  {INDEX_PATH:30s} ({index_size_kb:.1f} KB)")
print(f"  {METADATA_PATH:30s} (มีอยู่แล้ว)")
print()
print("ไฟล์ที่ต้องส่งให้ทีม RAG:")
print(f"  📦 {INDEX_PATH}    ← FAISS binary index")
print(f"  📄 {METADATA_PATH} ← ชื่อ/หมวด/สถานี/รายละเอียด")

บันทึกสำเร็จ:
  tourism_faiss.index            (216.0 KB)
  tourism_metadata.json          (มีอยู่แล้ว)

ไฟล์ที่ต้องส่งให้ทีม RAG:
  📦 tourism_faiss.index    ← FAISS binary index
  📄 tourism_metadata.json ← ชื่อ/หมวด/สถานี/รายละเอียด


## 6. ทดสอบโหลด Index ใหม่จากไฟล์

In [7]:
# จำลองการใช้งานของทีมอื่น — โหลดจากไฟล์เปล่า
loaded_index = faiss.read_index(INDEX_PATH)
with open(METADATA_PATH, encoding="utf-8") as f:
    loaded_metadata = json.load(f)

print(f"โหลด index สำเร็จ: {loaded_index.ntotal} vectors, dim={loaded_index.d}")
print(f"โหลด metadata สำเร็จ: {len(loaded_metadata)} รายการ")

# ยืนยันผลลัพธ์ตรงกับ index เดิม
s2, i2 = loaded_index.search(test_vec, k=1)
assert i2[0][0] == 0 and abs(s2[0][0] - 1.0) < 1e-4
print("✓ Index ที่โหลดมาให้ผลลัพธ์ตรงกัน")

โหลด index สำเร็จ: 54 vectors, dim=1024
โหลด metadata สำเร็จ: 54 รายการ
✓ Index ที่โหลดมาให้ผลลัพธ์ตรงกัน


## 7. RAG Retriever Function

In [8]:
def retrieve(
    query_embedding: np.ndarray,
    top_k: int = 5,
    score_threshold: float = 0.0,
) -> list[dict]:
    """
    ค้นหาสถานที่ท่องเที่ยวที่เกี่ยวข้องจาก FAISS index

    Args:
        query_embedding: numpy array shape (1, 1024) หรือ (1024,) — L2-normalized
        top_k          : จำนวนผลลัพธ์ที่ต้องการ
        score_threshold: กรอง document ที่ score ต่ำกว่าค่านี้ออก (0.0 = ไม่กรอง)

    Returns:
        list of dict ที่มี keys: name, category, station, details, context, score
        เรียงจาก score สูงสุด
    """
    vec = np.array(query_embedding, dtype=np.float32)
    if vec.ndim == 1:
        vec = vec.reshape(1, -1)
    faiss.normalize_L2(vec)

    scores, indices = loaded_index.search(vec, k=top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:  # FAISS ส่ง -1 เมื่อหา k ไม่ครบ
            continue
        if score < score_threshold:
            continue
        doc = loaded_metadata[idx].copy()
        doc["score"] = float(score)
        # context string สำหรับ inject เข้า LLM prompt
        doc["context"] = (
            f"[{doc['category']}] {doc['name']} ({doc['station']}):\n"
            f"{doc['details']}"
        )
        results.append(doc)

    return results


def format_rag_context(results: list[dict]) -> str:
    """รวม context ทุก chunk เป็น string เดียวสำหรับ inject เข้า system prompt"""
    chunks = [f"{i+1}. {r['context']}" for i, r in enumerate(results)]
    return "\n\n".join(chunks)


print("✓ RAG retriever functions พร้อมใช้งาน")

✓ RAG retriever functions พร้อมใช้งาน


## 8. ทดสอบ RAG Retriever

In [9]:
# ทดสอบโดยใช้ embedding ของ document ที่มีอยู่แทน (ไม่ต้องโหลด model ใหม่)
# ในการใช้งานจริง ทีม RAG จะ encode query ด้วย BAAI/bge-m3 เอง

# จำลอง query embedding (ใช้ vector ของ 'สวนเบญจกิติ')
park_idx = next(i for i, m in enumerate(loaded_metadata) if "สวนเบญจกิติ" in m["name"])
mock_query_vec = embeddings[park_idx]

results = retrieve(mock_query_vec, top_k=3, score_threshold=0.4)

print("=" * 60)
print("ตัวอย่าง RAG context (inject เข้า LLM prompt):")
print("=" * 60)
print(format_rag_context(results))
print("\n" + "=" * 60)
print(f"จำนวน chunks ที่ retrieve ได้: {len(results)}")
for r in results:
    print(f"  [score={r['score']:.4f}] {r['name']}")

ตัวอย่าง RAG context (inject เข้า LLM prompt):
1. [พื้นที่สีเขียว และจุดชมวิวเมือง] สวนเบญจกิติ (MRT สุขุมวิท/BTS อโศก):
สวนเบญจกิติ (MRT สุขุมวิท/BTS อโศก): สวนป่าเชิงนิเวศ (Forest Park) ขนาดกว่า 450 ไร่ เปิด 05:00 - 21:00 น.

2. [พื้นที่สีเขียว และจุดชมวิวเมือง] สวนเบญจสิริ (BTS พร้อมพงษ์):
สวนเบญจสิริ (BTS พร้อมพงษ์): สร้างปี 2535 พื้นที่ 47,000 ตร.ม. พร้อมประติมากรรม 12 ชิ้น เปิด 04:30 - 22:00 น.

3. [พื้นที่สีเขียว และจุดชมวิวเมือง] อุทยาน 100 ปี จุฬาฯ (BTS สยาม/สนามกีฬา):
อุทยาน 100 ปี จุฬาฯ (BTS สยาม/สนามกีฬา): สวนหน่วงน้ำที่ออกแบบเพื่อรองรับปัญหาน้ำท่วม เปิด 05:00 - 22:00 น.

จำนวน chunks ที่ retrieve ได้: 3
  [score=1.0000] สวนเบญจกิติ
  [score=0.7889] สวนเบญจสิริ
  [score=0.7425] อุทยาน 100 ปี จุฬาฯ


## 9. ตัวอย่างการใช้งาน

```python
# pip install faiss-cpu sentence-transformers
import os
from sentence_transformers import SentenceTransformer
import torch
import re
import json
from pathlib import Path

os.environ["HF_TOKEN"] = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"

import faiss, json, numpy as np
from sentence_transformers import SentenceTransformer

# 1. โหลด model เพื่อ encode query
model = SentenceTransformer("BAAI/bge-m3")

# 2. โหลด FAISS index และ metadata
index = faiss.read_index("tourism_faiss.index")
with open("tourism_metadata.json", encoding="utf-8") as f:
    metadata = json.load(f)

# 3. Encode query (ใช้ instruction prefix สำหรับ retrieval)
QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "
query = "วัดสวยใกล้ BTS"
query_vec = model.encode([QUERY_INSTRUCTION + query], normalize_embeddings=True)

# 4. Search
scores, indices = index.search(query_vec.astype(np.float32), k=5)

# 5. Build context สำหรับ LLM
context_chunks = []
for score, idx in zip(scores[0], indices[0]):
    if idx == -1: continue
    doc = metadata[idx]
    context_chunks.append(
        f"[{doc['category']}] {doc['name']} ({doc['station']}):\n{doc['details']}"
    )

rag_context = "\n\n".join(context_chunks)

# 6. ส่ง context + query เข้า LLM
system_prompt = f"""คุณเป็นผู้ช่วยแนะนำการท่องเที่ยวกรุงเทพ ตอบคำถามจากข้อมูลด้านล่างเท่านั้น

ข้อมูลสถานที่:
{rag_context}"""
```